In [83]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from scipy.io import arff
import warnings
warnings.filterwarnings('ignore')

# Load data
data = pd.read_csv('../../data/heart_disease.csv')
print("Dataset Shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())
print("\nData Info:")
print(data.info())
print("\nMissing Values:")
print(data.isnull().sum())

Dataset Shape: (10000, 21)

First 5 rows:
    Age  Gender  Blood Pressure  Cholesterol Level Exercise Habits Smoking  \
0  56.0    Male           153.0              155.0            High     Yes   
1  69.0  Female           146.0              286.0            High      No   
2  46.0    Male           126.0              216.0             Low      No   
3  32.0  Female           122.0              293.0            High     Yes   
4  60.0    Male           166.0              242.0             Low     Yes   

  Family Heart Disease Diabetes        BMI High Blood Pressure  ...  \
0                  Yes       No  24.991591                 Yes  ...   
1                  Yes      Yes  25.221799                  No  ...   
2                   No       No  29.855447                  No  ...   
3                  Yes       No  24.130477                 Yes  ...   
4                  Yes      Yes  20.486289                 Yes  ...   

  High LDL Cholesterol Alcohol Consumption Stress Level Sleep 

## Step 2: Handle Missing Values
We'll use appropriate imputation strategies:
- **Numeric columns**: Median imputation (robust to outliers)
- **Categorical columns**: Mode imputation (most frequent value)

In [84]:
# Identify numeric and categorical columns
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()

# Remove target column from preprocessing
target_col = 'Heart Disease Status'
if target_col in numeric_cols:
    numeric_cols.remove(target_col)
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

# Check missing values before imputation
missing_before = data.isnull().sum()
print(f"\nMissing values before imputation:")
print(missing_before[missing_before > 0])

Numeric columns (9): ['Age', 'Blood Pressure', 'Cholesterol Level', 'BMI', 'Sleep Hours', 'Triglyceride Level', 'Fasting Blood Sugar', 'CRP Level', 'Homocysteine Level']

Categorical columns (11): ['Gender', 'Exercise Habits', 'Smoking', 'Family Heart Disease', 'Diabetes', 'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol', 'Alcohol Consumption', 'Stress Level', 'Sugar Consumption']

Missing values before imputation:
Age                       29
Gender                    19
Blood Pressure            19
Cholesterol Level         30
Exercise Habits           25
Smoking                   25
Family Heart Disease      21
Diabetes                  30
BMI                       22
High Blood Pressure       26
Low HDL Cholesterol       25
High LDL Cholesterol      26
Alcohol Consumption     2586
Stress Level              22
Sleep Hours               25
Sugar Consumption         30
Triglyceride Level        26
Fasting Blood Sugar       22
CRP Level                 26
Homocystei

In [85]:
# Handle missing values - Numeric columns with median
if numeric_cols:
    num_imputer = SimpleImputer(strategy='median')
    data[numeric_cols] = num_imputer.fit_transform(data[numeric_cols])
    print("Numeric columns imputed with median")

# Handle missing values - Categorical columns with mode
if categorical_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    data[categorical_cols] = cat_imputer.fit_transform(data[categorical_cols])
    print("Categorical columns imputed with mode")

# Verify no missing values remain
missing_after = data.isnull().sum()
print(f"\nMissing values after imputation: {missing_after.sum()}")
print("\nData shape after imputation:", data.shape)

Numeric columns imputed with median
Categorical columns imputed with mode

Missing values after imputation: 0

Data shape after imputation: (10000, 21)


## Step 3: Feature Engineering
Create domain-specific features to improve model performance:
1. **BMI Categories**: Underweight, Normal, Overweight, Obese
2. **Age Groups**: Young adult, Middle age, Senior, Elderly
3. **Risk Score**: Combined risk factors
4. **Exercise-Smoking Interaction**: Lifestyle combination patterns

In [86]:
# 1. BMI Categories
if 'BMI' in data.columns:
    data['BMI_Category'] = pd.cut(data['BMI'], 
                                    bins=[0, 18.5, 24.9, 29.9, 100],
                                    labels=['Underweight', 'Normal', 'Overweight', 'Obese'])
    print("Created BMI_Category feature")
    print(data['BMI_Category'].value_counts())

# 2. Age Groups
if 'Age' in data.columns:
    data['Age_Group'] = pd.cut(data['Age'], 
                                bins=[0, 35, 50, 65, 100],
                                labels=['Young_Adult', 'Middle_Age', 'Senior', 'Elderly'])
    print("\nCreated Age_Group feature")
    print(data['Age_Group'].value_counts())

# 3. Blood Pressure Category
if 'Blood Pressure (mmHg)' in data.columns:
    data['BP_Category'] = pd.cut(data['Blood Pressure (mmHg)'], 
                                  bins=[0, 120, 130, 140, 300],
                                  labels=['Normal', 'Elevated', 'High', 'Very_High'])
    print("\nCreated BP_Category feature")
    print(data['BP_Category'].value_counts())

# 4. Cholesterol Level Category
if 'Cholesterol (mg/dL)' in data.columns:
    data['Cholesterol_Category'] = pd.cut(data['Cholesterol (mg/dL)'], 
                                          bins=[0, 200, 240, 500],
                                          labels=['Desirable', 'Borderline', 'High'])
    print("\nCreated Cholesterol_Category feature")
    print(data['Cholesterol_Category'].value_counts())

Created BMI_Category feature
BMI_Category
Obese          4622
Normal         2776
Overweight     2370
Underweight     232
Name: count, dtype: int64

Created Age_Group feature
Age_Group
Young_Adult    2785
Elderly        2446
Middle_Age     2420
Senior         2349
Name: count, dtype: int64


In [87]:
# 5. Risk Score - Combined risk factors
risk_score = 0

# Add points for risk factors
if 'Smoking' in data.columns:
    risk_score += (data['Smoking'] == 'Yes').astype(int) * 2

if 'Diabetes' in data.columns:
    risk_score += (data['Diabetes'] == 'Yes').astype(int) * 2

if 'Family History' in data.columns:
    risk_score += (data['Family History'] == 'Yes').astype(int) * 1

if 'Exercise Hours Per Week' in data.columns:
    risk_score += (data['Exercise Hours Per Week'] < 2).astype(int) * 1

if 'BMI' in data.columns:
    risk_score += (data['BMI'] >= 30).astype(int) * 2

if 'Blood Pressure (mmHg)' in data.columns:
    risk_score += (data['Blood Pressure (mmHg)'] >= 140).astype(int) * 2

data['Risk_Score'] = risk_score
print("\nCreated Risk_Score feature")
print(f"Risk Score range: {data['Risk_Score'].min()} - {data['Risk_Score'].max()}")
print(data['Risk_Score'].value_counts().sort_index())


Created Risk_Score feature
Risk Score range: 0 - 6
Risk_Score
0    1302
2    3848
4    3719
6    1131
Name: count, dtype: int64


In [88]:
# 6. Lifestyle Interaction Features
if 'Smoking' in data.columns and 'Exercise Hours Per Week' in data.columns:
    # Poor lifestyle: Smoking + Low exercise
    data['Poor_Lifestyle'] = ((data['Smoking'] == 'Yes') & 
                              (data['Exercise Hours Per Week'] < 2)).astype(int)
    print("\nCreated Poor_Lifestyle feature")
    print(f"Poor lifestyle count: {data['Poor_Lifestyle'].sum()}")

if 'Diet' in data.columns and 'Exercise Hours Per Week' in data.columns:
    # Healthy lifestyle: Healthy diet + Regular exercise
    data['Healthy_Lifestyle'] = ((data['Diet'] == 'Healthy') & 
                                 (data['Exercise Hours Per Week'] >= 4)).astype(int)
    print("Created Healthy_Lifestyle feature")
    print(f"Healthy lifestyle count: {data['Healthy_Lifestyle'].sum()}")

# 7. High Risk Age-BMI combination
if 'Age' in data.columns and 'BMI' in data.columns:
    data['High_Risk_Age_BMI'] = ((data['Age'] >= 60) & (data['BMI'] >= 30)).astype(int)
    print("Created High_Risk_Age_BMI feature")
    print(f"High risk age-BMI count: {data['High_Risk_Age_BMI'].sum()}")

print(f"\nFeature engineering complete!")
print(f"Total features now: {data.shape[1]}")

Created High_Risk_Age_BMI feature
High risk age-BMI count: 1556

Feature engineering complete!
Total features now: 25


## Step 4: Data Summary and Statistics
Review the processed dataset with new engineered features

In [89]:
print("=" * 60)
print("PREPROCESSED DATASET SUMMARY")
print("=" * 60)
print(f"\nShape: {data.shape}")
print(f"Total Features: {data.shape[1]}")
print(f"Total Instances: {data.shape[0]}")
print(f"\nMissing Values: {data.isnull().sum().sum()}")

print("\n" + "=" * 60)
print("NUMERIC FEATURES STATISTICS")
print("=" * 60)
print(data.describe())

print("\n" + "=" * 60)
print("CATEGORICAL FEATURES")
print("=" * 60)
categorical_features = data.select_dtypes(include=['object', 'category']).columns
for col in categorical_features:
    print(f"\n{col}:")
    print(data[col].value_counts())

print("\n" + "=" * 60)
print("NEW ENGINEERED FEATURES")
print("=" * 60)
engineered_features = ['BMI_Category', 'Age_Group', 'BP_Category', 'Cholesterol_Category', 
                       'Risk_Score', 'Poor_Lifestyle', 'Healthy_Lifestyle', 'High_Risk_Age_BMI']
for col in engineered_features:
    if col in data.columns:
        print(f"✓ {col}")

PREPROCESSED DATASET SUMMARY

Shape: (10000, 25)
Total Features: 25
Total Instances: 10000

Missing Values: 0

NUMERIC FEATURES STATISTICS
                Age  Blood Pressure  Cholesterol Level           BMI  \
count  10000.000000    10000.000000       10000.000000  10000.000000   
mean      49.295400      149.758200         225.427300     29.077274   
std       18.167574       17.556268          43.510401      6.300156   
min       18.000000      120.000000         150.000000     18.002837   
25%       34.000000      134.000000         187.000000     23.668887   
50%       49.000000      150.000000         226.000000     29.079492   
75%       65.000000      165.000000         263.000000     34.509009   
max       80.000000      180.000000         300.000000     39.996954   

        Sleep Hours  Triglyceride Level  Fasting Blood Sugar     CRP Level  \
count  10000.000000        10000.000000         10000.000000  10000.000000   
mean       6.991359          250.732500           120.14

In [90]:
# Ensure 'Heart Disease Status' is the target and placed last
target_col = 'Heart Disease Status'
if target_col in data.columns:
    # Reorder columns to put target last
    cols = [col for col in data.columns if col != target_col]
    cols.append(target_col)
    data = data[cols]
    print(f"Target column '{target_col}' positioned as last column")
else:
    print(f"Warning: Target column '{target_col}' not found in dataset")

# Save to CSV
data.to_csv('../../data/heart_disease_feature_engineered.csv', index=False)

Target column 'Heart Disease Status' positioned as last column


## Step 5: Save Preprocessed Data to ARFF
Save the preprocessed dataset with engineered features for Weka analysis.

**Note:** Categorical features are kept as nominal (not encoded) for optimal performance with tree-based classifiers (J48, RandomForest, NaiveBayes).